<a href="https://colab.research.google.com/github/memyselfandglitch/TransformerFromScratch/blob/main/GoogleColab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
# If Colab has enough GPU memory, try:
# model_id = "Qwen/Qwen2.5-1.5B-Instruct"
# model_id = "Qwen/Qwen2.5-7B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)

model.eval()


device: cuda


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [ ]:
print("model_type:", model.config.model_type)
print("layers:", model.config.num_hidden_layers)
print("attention heads:", model.config.num_attention_heads)
print("kv heads:", model.config.num_key_value_heads)
print("hidden size:", model.config.hidden_size)

head_dim = model.config.hidden_size // model.config.num_attention_heads
print("head_dim:", head_dim)


model_type: qwen2
layers: 24
attention heads: 14
kv heads: 2
hidden size: 896
head_dim: 64


In [ ]:
prompt = "The red cat was very"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs, use_cache=True)

past = outputs.past_key_values

print(type(past))
print(type(past.layers[0]) if hasattr(past, "layers") else type(past[0]))


<class 'transformers.cache_utils.DynamicCache'>
<class 'transformers.cache_utils.DynamicLayer'>


In [ ]:
import torch

def extract_kv_layers(past):
    kv_layers = []

    for i, item in enumerate(past):
        if not isinstance(item, tuple):
            raise TypeError(f"Layer {i} cache item is not a tuple: {type(item)}")

        if len(item) < 2:
            raise TypeError(f"Layer {i} cache tuple is too short: len={len(item)}")

        k = item[0]
        v = item[1]

        if not isinstance(k, torch.Tensor):
            raise TypeError(f"Layer {i} K is not a tensor: {type(k)}")

        if not isinstance(v, torch.Tensor):
            raise TypeError(f"Layer {i} V is not a tensor: {type(v)}")

        # Ignore any extra metadata after K and V.
        kv_layers.append((k, v))

    return kv_layers


def show_kv_cache(label, past):
    kv_layers = extract_kv_layers(past)

    k0, v0 = kv_layers[0]

    print(f"\n{label}")
    print("number of cached layers:", len(kv_layers))

    print("Layer 0 K shape:", tuple(k0.shape))
    print("Layer 0 V shape:", tuple(v0.shape))

    print("Layer 0 K stride:", k0.stride())
    print("Layer 0 V stride:", v0.stride())

    print("Layer 0 K contiguous:", k0.is_contiguous())
    print("Layer 0 V contiguous:", v0.is_contiguous())

    total_bytes = sum(
        k.numel() * k.element_size() + v.numel() * v.element_size()
        for k, v in kv_layers
    )

    print("Total KV cache:", f"{total_bytes / 1024**2:.2f} MiB")


In [ ]:
show_kv_cache("After prompt prefill", past)



After prompt prefill
number of cached layers: 24
Layer 0 K shape: (1, 2, 5, 64)
Layer 0 V shape: (1, 2, 5, 64)
Layer 0 K stride: (640, 320, 64, 1)
Layer 0 V stride: (640, 320, 64, 1)
Layer 0 K contiguous: True
Layer 0 V contiguous: True
Total KV cache: 0.06 MiB


In [ ]:
seq_lengths = [5, 128, 256, 512, 1024, 2048, 4096, 8192]

print(f"{'seq_len':>8} | {'KV cache MiB':>14} | {'Layer 0 K shape':>22}")
print("-" * 55)

for seq_len in seq_lengths:
    try:
        input_ids = torch.randint(
            low=100,
            high=min(1000, tokenizer.vocab_size),
            size=(1, seq_len),
            device=model.device,
        )

        with torch.no_grad():
            outputs = model(input_ids=input_ids, use_cache=True)

        past = outputs.past_key_values
        kv_layers = extract_kv_layers(past)
        k0, v0 = kv_layers[0]

        total_mib = sum(
            k.numel() * k.element_size() + v.numel() * v.element_size()
            for k, v in kv_layers
        ) / 1024**2

        print(f"{seq_len:8d} | {total_mib:14.2f} | {str(tuple(k0.shape)):>22}")

        del input_ids, outputs, past
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"{seq_len:8d} | FAILED: {e}")
        break


 seq_len |   KV cache MiB |        Layer 0 K shape
-------------------------------------------------------
       5 |           0.06 |          (1, 2, 5, 64)
     128 |           1.50 |        (1, 2, 128, 64)
     256 |           3.00 |        (1, 2, 256, 64)
     512 |           6.00 |        (1, 2, 512, 64)
    1024 |          12.00 |       (1, 2, 1024, 64)
    2048 |          24.00 |       (1, 2, 2048, 64)
    4096 |          48.00 |       (1, 2, 4096, 64)
    8192 | FAILED: CUDA out of memory. Tried to allocate 3.50 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.80 GiB is free. Including non-PyTorch memory, this process has 12.76 GiB memory in use. Of the allocated memory 12.59 GiB is allocated by PyTorch, and 40.27 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cu